In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import (
    shapiro,
    pearsonr,
    spearmanr,
    mannwhitneyu,
    ttest_ind,
    ttest_rel,
    wilcoxon,
    chi2_contingency
)
MAIN_FOLDER = "../"
PEDERSON_FILE = "6-right_left_pederson_agreement-2/"

In [ ]:
# ============================================================
# 2. Load dataset
# ============================================================
FILE_NAME = "paired_list"

# Read the excel file
paired = pd.read_excel("../../"+FILE_NAME+".xlsx")

# Show the head
print(paired.tail())

In [ ]:
# ============================================================
# 2. Clean Pederson columns
# ============================================================

# Convert Pederson values to numeric
paired["pederson_left"] = pd.to_numeric(paired["pederson_left"], errors="coerce")
paired["pederson_right"] = pd.to_numeric(paired["pederson_right"], errors="coerce")

# Keep only patients with both left and right Pederson scores
ped_df = paired.dropna(subset=["pederson_left", "pederson_right"]).copy()

print("Number of patients with bilateral Pederson data:", len(ped_df))

In [ ]:
# Create output folder
output_folder = MAIN_FOLDER+PEDERSON_FILE
os.makedirs(output_folder, exist_ok=True)

In [ ]:
# ============================================================
# 3. Exact Pederson score agreement
# ============================================================

# Check whether left and right Pederson scores are exactly the same
ped_df["pederson_exact_match"] = (
    ped_df["pederson_left"] == ped_df["pederson_right"]
)

total_patients = len(ped_df)
matched_patients = ped_df["pederson_exact_match"].sum()
unmatched_patients = total_patients - matched_patients

match_percentage = matched_patients / total_patients * 100
unmatch_percentage = unmatched_patients / total_patients * 100

exact_match_summary = pd.DataFrame({
    "Metric": [
        "Total patients with bilateral Pederson data",
        "Exact matched patients",
        "Unmatched patients",
        "Exact match percentage",
        "Unmatched percentage"
    ],
    "Value": [
        total_patients,
        matched_patients,
        unmatched_patients,
        round(match_percentage, 2),
        round(unmatch_percentage, 2)
    ]
})

print("\nExact Pederson Score Agreement Summary")
print(exact_match_summary)

exact_match_summary.to_excel(
    os.path.join(output_folder, "pederson_exact_match_summary.xlsx"),
    index=False
)

In [ ]:
pederson_crosstab = pd.crosstab(
    ped_df["pederson_left"],
    ped_df["pederson_right"]
)

pederson_crosstab_percent = pd.crosstab(
    ped_df["pederson_left"],
    ped_df["pederson_right"],
    normalize="index"
) * 100

print("\nLeft vs Right Pederson Score Crosstab")
print(pederson_crosstab)

print("\nLeft vs Right Pederson Score Crosstab (%)")
print(pederson_crosstab_percent.round(2))

pederson_crosstab.to_excel(
    os.path.join(output_folder, "left_right_pederson_score_crosstab.xlsx")
)

pederson_crosstab_percent.round(2).to_excel(
    os.path.join(output_folder, "left_right_pederson_score_crosstab_percent.xlsx")
)

In [ ]:
from sklearn.metrics import cohen_kappa_score

# Cohen's Kappa for exact categorical agreement
kappa = cohen_kappa_score(
    ped_df["pederson_left"],
    ped_df["pederson_right"]
)

# Weighted Cohen's Kappa for ordinal agreement
weighted_kappa_linear = cohen_kappa_score(
    ped_df["pederson_left"],
    ped_df["pederson_right"],
    weights="linear"
)

weighted_kappa_quadratic = cohen_kappa_score(
    ped_df["pederson_left"],
    ped_df["pederson_right"],
    weights="quadratic"
)

kappa_summary = pd.DataFrame({
    "Metric": [
        "Cohen's Kappa",
        "Linear Weighted Cohen's Kappa",
        "Quadratic Weighted Cohen's Kappa"
    ],
    "Value": [
        round(kappa, 4),
        round(weighted_kappa_linear, 4),
        round(weighted_kappa_quadratic, 4)
    ]
})

print("\nPederson Agreement Kappa Statistics")
print(kappa_summary)

kappa_summary.to_excel(
    os.path.join(output_folder, "pederson_kappa_statistics.xlsx"),
    index=False
)

In [ ]:
pederson_order = list(range(3, 11))  # 3, 4, 5, 6, 7, 8, 9, 10


pederson_crosstab = pd.crosstab(
    ped_df["pederson_left"],
    ped_df["pederson_right"]
).reindex(
    index=pederson_order,
    columns=pederson_order,
    fill_value=0
)

pederson_crosstab_percent = pd.crosstab(
    ped_df["pederson_left"],
    ped_df["pederson_right"],
    normalize="index"
) * 100

print("\nLeft vs Right Pederson Score Crosstab")
print(pederson_crosstab)

print("\nLeft vs Right Pederson Score Crosstab (%)")
print(pederson_crosstab_percent.round(2))

pederson_crosstab.to_excel(
    os.path.join(output_folder, "left_right_pederson_score_crosstab.xlsx")
)

pederson_crosstab_percent.round(2).to_excel(
    os.path.join(output_folder, "left_right_pederson_score_crosstab_percent.xlsx")
)

In [ ]:
# Cohen's Kappa for exact categorical agreement
kappa = cohen_kappa_score(
    ped_df["pederson_left"],
    ped_df["pederson_right"]
)

# Weighted Cohen's Kappa for ordinal agreement
weighted_kappa_linear = cohen_kappa_score(
    ped_df["pederson_left"],
    ped_df["pederson_right"],
    weights="linear"
)

weighted_kappa_quadratic = cohen_kappa_score(
    ped_df["pederson_left"],
    ped_df["pederson_right"],
    weights="quadratic"
)

kappa_summary = pd.DataFrame({
    "Metric": [
        "Cohen's Kappa",
        "Linear Weighted Cohen's Kappa",
        "Quadratic Weighted Cohen's Kappa"
    ],
    "Value": [
        round(kappa, 4),
        round(weighted_kappa_linear, 4),
        round(weighted_kappa_quadratic, 4)
    ]
})

print("\nPederson Agreement Kappa Statistics")
print(kappa_summary)

kappa_summary.to_excel(
    os.path.join(output_folder, "pederson_kappa_statistics.xlsx"),
    index=False
)

In [ ]:
# ============================================================
# 4. Convert Pederson scores into difficulty groups
# ============================================================

def pederson_difficulty(score):
    """
    Convert Pederson score into difficulty category.

    Common grouping:
    3-4  : Easy
    5-6  : Moderate
    7-10 : Difficult
    """
    if pd.isna(score):
        return np.nan
    elif score <= 4:
        return "Easy"
    elif score <= 6:
        return "Moderate"
    else:
        return "Difficult"


ped_df["pederson_difficulty_left"] = ped_df["pederson_left"].apply(pederson_difficulty)
ped_df["pederson_difficulty_right"] = ped_df["pederson_right"].apply(pederson_difficulty)

# Check whether left and right difficulty groups are the same
ped_df["pederson_difficulty_match"] = (
    ped_df["pederson_difficulty_left"] == ped_df["pederson_difficulty_right"]
)

difficulty_total = len(ped_df)
difficulty_matched = ped_df["pederson_difficulty_match"].sum()
difficulty_unmatched = difficulty_total - difficulty_matched

difficulty_match_percentage = difficulty_matched / difficulty_total * 100
difficulty_unmatch_percentage = difficulty_unmatched / difficulty_total * 100

difficulty_match_summary = pd.DataFrame({
    "Metric": [
        "Total patients with bilateral Pederson data",
        "Matched difficulty group patients",
        "Unmatched difficulty group patients",
        "Difficulty group match percentage",
        "Difficulty group unmatched percentage"
    ],
    "Value": [
        difficulty_total,
        difficulty_matched,
        difficulty_unmatched,
        round(difficulty_match_percentage, 2),
        round(difficulty_unmatch_percentage, 2)
    ]
})

print("\nPederson Difficulty Group Agreement Summary")
print(difficulty_match_summary)

difficulty_match_summary.to_excel(
    os.path.join(output_folder, "pederson_difficulty_group_match_summary.xlsx"),
    index=False
)

In [ ]:
difficulty_order = ["Easy", "Moderate", "Difficult"]

difficulty_crosstab = pd.crosstab(
    ped_df["pederson_difficulty_left"],
    ped_df["pederson_difficulty_right"]
).reindex(index=difficulty_order, columns=difficulty_order, fill_value=0)

difficulty_crosstab_percent = pd.crosstab(
    ped_df["pederson_difficulty_left"],
    ped_df["pederson_difficulty_right"],
    normalize="index"
).reindex(index=difficulty_order, columns=difficulty_order, fill_value=0) * 100

print("\nLeft vs Right Pederson Difficulty Group Crosstab")
print(difficulty_crosstab)

print("\nLeft vs Right Pederson Difficulty Group Crosstab (%)")
print(difficulty_crosstab_percent.round(2))

difficulty_crosstab.to_excel(
    os.path.join(output_folder, "left_right_pederson_difficulty_crosstab.xlsx")
)

difficulty_crosstab_percent.round(2).to_excel(
    os.path.join(output_folder, "left_right_pederson_difficulty_crosstab_percent.xlsx")
)

In [ ]:
difficulty_kappa = cohen_kappa_score(
    ped_df["pederson_difficulty_left"],
    ped_df["pederson_difficulty_right"]
)

difficulty_weighted_kappa = cohen_kappa_score(
    ped_df["pederson_difficulty_left"],
    ped_df["pederson_difficulty_right"],
    weights="linear"
)

difficulty_kappa_summary = pd.DataFrame({
    "Metric": [
        "Cohen's Kappa for Pederson difficulty groups",
        "Linear Weighted Cohen's Kappa for Pederson difficulty groups"
    ],
    "Value": [
        round(difficulty_kappa, 4),
        round(difficulty_weighted_kappa, 4)
    ]
})

print("\nPederson Difficulty Group Kappa Statistics")
print(difficulty_kappa_summary)

difficulty_kappa_summary.to_excel(
    os.path.join(output_folder, "pederson_difficulty_group_kappa_statistics.xlsx"),
    index=False
)

In [ ]:
match_counts = (
    ped_df["pederson_exact_match"]
    .value_counts()
    .reindex([True, False], fill_value=0)
)

match_counts.index = ["Exact Match", "No Exact Match"]

def my_autopct(pct):
    return ("%1.1f%%" % pct) if pct > 1.0 else ""

# ---------------------------
# Global style settings for academic appearance
# ---------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif", "serif"],
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "figure.dpi": 300,
    "savefig.dpi": 300
})

plt.figure(figsize=(8, 8))

colors = plt.cm.Blues(np.linspace(0.45, 0.85, len(match_counts)))

wedges, texts, autotexts = plt.pie(
    match_counts,
    autopct=my_autopct,
    startangle=90,
    pctdistance=0.85,
    colors=colors,
    wedgeprops=dict(width=0.5, edgecolor="w")
)


plt.title("Exact Match Rate of Left and Right Pederson Scores")


# Set percentage text style
for autotext in autotexts:
    autotext.set_color("white")
    autotext.set_fontsize(12)
    autotext.set_fontweight("bold")

plt.legend(
    wedges,
    match_counts.index.astype(str),
    title="Match Status",
    loc="center left",
    bbox_to_anchor=(1, 0, 0.5, 1)
)

plt.tight_layout()

plt.savefig(
    os.path.join(output_folder, "pederson_exact_match_donut_chart.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.colors as mcolors

# ---------------------------
# Global style settings for academic appearance
# ---------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif", "serif"],
    "font.size": 15,
    "axes.titlesize": 15,
    "axes.labelsize": 15,
    "xtick.labelsize": 15,
    "ytick.labelsize": 15,
    "legend.fontsize": 15,
    "figure.dpi": 300,
    "savefig.dpi": 300
})


plt.figure(figsize=(8, 6))

# ---------------------------
# Create a darker blue colormap
# ---------------------------
base_cmap = plt.cm.Blues

# Use only the medium-to-dark part of the Blues colormap
# 0.25 avoids very pale blue/white colors
# 0.90 avoids extremely dark colors
blue_colors = base_cmap(np.linspace(0.15, 0.95, 256))

custom_blues = mcolors.LinearSegmentedColormap.from_list(
    "custom_blues",
    blue_colors
)

img = plt.imshow(pederson_crosstab, aspect="equal", cmap=custom_blues)

#img = plt.imshow(pederson_crosstab, aspect="auto", cmap="magma")

#plt.title("Left vs Right Pederson Score Cross-tabulation")
plt.title("(d) Pederson score")
plt.xlabel("Right side")
plt.ylabel("Left side")

plt.xticks(
    ticks=np.arange(len(pederson_crosstab.columns)),
    labels=pederson_crosstab.columns,
    rotation=0,
    ha="center"
)

plt.yticks(
    ticks=np.arange(len(pederson_crosstab.index)),
    labels=pederson_crosstab.index
)


# Threshold controls when text becomes white.
threshold = pederson_crosstab.values.max() * 0.45

for i in range(len(pederson_crosstab.index)):
    for j in range(len(pederson_crosstab.columns)):
        value = pederson_crosstab.iloc[i, j]

        
        # Use white text on darker cells, dark text on lighter cells
        text_color = "white" if value >= threshold else "#222222"
        
        plt.text(
            j,
            i,
            str(value),
            ha="center",
            va="center",
            color=text_color,
            fontsize=15,
            fontweight="bold"
        )

plt.colorbar(img, label="Number of Patients")
plt.tight_layout()

plt.savefig(
    os.path.join(output_folder, "left_right_pederson_score_heatmap.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
difficulty_match_counts = ped_df["pederson_difficulty_match"].value_counts()

difficulty_match_counts.index = difficulty_match_counts.index.map({
    True: "Matched Difficulty Group",
    False: "Unmatched Difficulty Group"
})


# ---------------------------
# Global style settings for academic appearance
# ---------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif", "serif"],
    "font.size": 15,
    "axes.titlesize": 15,
    "axes.labelsize": 15,
    "xtick.labelsize": 15,
    "ytick.labelsize": 15,
    "legend.fontsize": 15,
    "figure.dpi": 300,
    "savefig.dpi": 300
})

plt.figure(figsize=(8, 8))

#colors = plt.cm.Pastel1(np.linspace(0, 1, len(difficulty_match_counts)))
colors = plt.cm.Blues(np.linspace(0.45, 0.85, len(difficulty_match_counts)))




wedges, texts, autotexts = plt.pie(
    difficulty_match_counts,
    autopct=my_autopct,
    startangle=90,
    pctdistance=0.85,
    colors=colors,
    wedgeprops=dict(width=0.5, edgecolor="w")
)

plt.title("Agreement of Left and Right Pederson Difficulty Groups")

# Set percentage text style
for autotext in autotexts:
    autotext.set_color("white")
    autotext.set_fontsize(15)
    autotext.set_fontweight("bold")

plt.legend(
    wedges,
    difficulty_match_counts.index.astype(str),
    title="Difficulty Group Agreement",
    loc="center left",
    bbox_to_anchor=(1, 0, 0.5, 1)
)

plt.tight_layout()

plt.savefig(
    os.path.join(output_folder, "pederson_difficulty_group_match_donut_chart.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ---------------------------
# Global style settings for academic appearance
# ---------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif", "serif"],
    "font.size": 15,
    "axes.titlesize": 15,
    "axes.labelsize": 15,
    "xtick.labelsize": 15,
    "ytick.labelsize": 15,
    "legend.fontsize": 15,
    "figure.dpi": 300,
    "savefig.dpi": 300
})

plt.figure(figsize=(7, 6))

# ---------------------------
# Create a darker blue colormap
# ---------------------------
base_cmap = plt.cm.Blues

# Use only the medium-to-dark part of the Blues colormap
# 0.25 avoids very pale blue/white colors
# 0.90 avoids extremely dark colors
blue_colors = base_cmap(np.linspace(0.15, 0.95, 256))

custom_blues = mcolors.LinearSegmentedColormap.from_list(
    "custom_blues",
    blue_colors
)

img = plt.imshow(difficulty_crosstab, aspect="equal", cmap=custom_blues)
#img = plt.imshow(difficulty_crosstab, aspect="auto", cmap="magma")

plt.title("(c) Pederson difficulty group")
plt.xlabel("Right side")
plt.ylabel("Left side")

plt.xticks(
    ticks=np.arange(len(difficulty_crosstab.columns)),
    labels=difficulty_crosstab.columns,
    rotation=0,
    ha="center"
)

plt.yticks(
    ticks=np.arange(len(difficulty_crosstab.index)),
    labels=difficulty_crosstab.index
)

# Threshold controls when text becomes white.
threshold = difficulty_crosstab.values.max() * 0.45

for i in range(len(difficulty_crosstab.index)):
    for j in range(len(difficulty_crosstab.columns)):
        value = difficulty_crosstab.iloc[i, j]

        # Use white text on darker cells, dark text on lighter cells
        text_color = "white" if value >= threshold else "#222222"
        
        plt.text(
            j,
            i,
            str(value),
            ha="center",
            va="center",
            color=text_color,
            fontsize=15,
            fontweight="bold"
        )

plt.colorbar(img, label="Number of Patients")
plt.tight_layout()

plt.savefig(
    os.path.join(output_folder, "left_right_pederson_difficulty_heatmap.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Standardize column names
paired.columns = paired.columns.str.strip().str.lower()

# Create output folder
output_folder = "pell_gregory_matching_outputs"
os.makedirs(output_folder, exist_ok=True)

# Clean Pell-Gregory columns
paired["pell_gregory_left"] = (
    paired["pell_gregory_left"]
    .astype(str)
    .str.strip()
    .str.upper()
)

paired["pell_gregory_right"] = (
    paired["pell_gregory_right"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Replace text-like missing values with NaN
missing_values = ["", "NAN", "NONE", "NULL", "YOK"]

paired["pell_gregory_left"] = paired["pell_gregory_left"].replace(missing_values, np.nan)
paired["pell_gregory_right"] = paired["pell_gregory_right"].replace(missing_values, np.nan)

# Keep only patients with both left and right Pell-Gregory values
pell_df = paired.dropna(subset=["pell_gregory_left", "pell_gregory_right"]).copy()

# Check exact left-right Pell-Gregory match
pell_df["pell_gregory_exact_match"] = (
    pell_df["pell_gregory_left"] == pell_df["pell_gregory_right"]
)

In [ ]:
pell_df.tail()

In [ ]:
# Calculate match statistics
total_patients = len(pell_df)
matched_patients = pell_df["pell_gregory_exact_match"].sum()
unmatched_patients = total_patients - matched_patients

match_percentage = matched_patients / total_patients * 100
unmatch_percentage = unmatched_patients / total_patients * 100

summary = pd.DataFrame({
    "Metric": [
        "Total patients with bilateral Pell-Gregory data",
        "Exact matched patients",
        "Unmatched patients",
        "Exact match percentage",
        "Unmatched percentage"
    ],
    "Value": [
        total_patients,
        matched_patients,
        unmatched_patients,
        round(match_percentage, 2),
        round(unmatch_percentage, 2)
    ]
})

summary

In [ ]:
match_counts = pell_df["pell_gregory_exact_match"].value_counts()

match_counts.index = match_counts.index.map({
    True: "Exact Match",
    False: "No Exact Match"
})

match_table = pd.DataFrame({
    "Count": match_counts,
    "Percentage (%)": (match_counts / match_counts.sum() * 100).round(2)
})

print(match_table)

match_table.to_excel(
    os.path.join(output_folder, "pell_gregory_exact_match_summary.xlsx")
)

In [ ]:
pell_crosstab = pd.crosstab(
    pell_df["pell_gregory_left"],
    pell_df["pell_gregory_right"]
)

pell_crosstab_percent = pd.crosstab(
    pell_df["pell_gregory_left"],
    pell_df["pell_gregory_right"],
    normalize="index"
) * 100

print("Left vs Right Pell-Gregory Crosstab")
print(pell_crosstab)

print("Left vs Right Pell-Gregory Crosstab (%)")
print(pell_crosstab_percent.round(2))

pell_crosstab.to_excel(
    os.path.join(output_folder, "left_right_pell_gregory_crosstab.xlsx")
)

pell_crosstab_percent.round(2).to_excel(
    os.path.join(output_folder, "left_right_pell_gregory_crosstab_percent.xlsx")
)

In [ ]:
from sklearn.metrics import cohen_kappa_score

kappa = cohen_kappa_score(
    pell_df["pell_gregory_left"],
    pell_df["pell_gregory_right"]
)

kappa_result = pd.DataFrame({
    "Metric": ["Cohen's Kappa"],
    "Value": [round(kappa, 4)]
})

print(kappa_result)

kappa_result.to_excel(
    os.path.join(output_folder, "pell_gregory_cohens_kappa.xlsx"),
    index=False
)

In [ ]:
def my_autopct(pct):
    return ("%1.1f%%" % pct) if pct > 1.0 else ""

plt.figure(figsize=(8, 8))

colors = plt.cm.Pastel1(np.linspace(0, 1, len(match_counts)))

wedges, texts, autotexts = plt.pie(
    match_counts,
    autopct=my_autopct,
    startangle=90,
    pctdistance=0.85,
    colors=colors,
    wedgeprops=dict(width=0.5, edgecolor="w")
)

plt.title("Exact Match Rate of Left and Right Pell-Gregory Classes")

plt.legend(
    wedges,
    match_counts.index.astype(str),
    title="Match Status",
    loc="center left",
    bbox_to_anchor=(1, 0, 0.5, 1)
)

plt.tight_layout()

plt.savefig(
    os.path.join(output_folder, "pell_gregory_exact_match_donut_chart.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(figsize=(8, 6))

img = plt.imshow(pell_crosstab, aspect="auto", cmap="magma")

plt.title("Left vs Right Pell-Gregory Cross-tabulation")
plt.xlabel("Right Pell-Gregory Class")
plt.ylabel("Left Pell-Gregory Class")

plt.xticks(
    ticks=np.arange(len(pell_crosstab.columns)),
    labels=pell_crosstab.columns,
    rotation=30,
    ha="right"
)

plt.yticks(
    ticks=np.arange(len(pell_crosstab.index)),
    labels=pell_crosstab.index
)

for i in range(len(pell_crosstab.index)):
    for j in range(len(pell_crosstab.columns)):
        value = pell_crosstab.iloc[i, j]
        plt.text(
            j,
            i,
            str(value),
            ha="center",
            va="center",
            color="white",
            fontsize=10,
            fontweight="bold"
        )

plt.colorbar(img, label="Number of Patients")
plt.tight_layout()

plt.savefig(
    os.path.join(output_folder, "left_right_pell_gregory_crosstab_heatmap.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()